# European Option Pricing with Discrete Cash Dividends (`'forward'` / `'escrowed'`)

This notebook prices European Call and Put options on **KO (Coca-Cola)** when the company pays **fixed cash dividends on known dates** (for example $\$0.46$ per share every quarter).

**Why a new model?** A cash dividend makes the stock price **jump down** by exactly that amount on the ex-date, so the price is no longer lognormal and plain Black-Scholes does not apply exactly. Two classic approximations handle this, and this notebook implements **both** behind one switch:

| `dividend_model` | What gets adjusted | Formula |
|---|---|---|
| `'escrowed'` | the **stock price** | $S^* = S_0 - PV(D)$ |
| `'forward'` | the **strike** | $K^* = K + FV(D)$ |

**Plan:**
1. fetch KO data + risk-free rate
2. historical volatility from log returns
3. option parameters
4. build the dividend schedule and calculate $PV(D)$, $FV(D)$
5. price with both models (switch `dividend_model`) + put-call parity check
6. implied volatility under both models

## Input

| Input | Symbol | Where we get it |
|---|---|---|
| Current stock price | $S_0$ | Market (yfinance, latest close) |
| Strike price | $K$ | Option contract |
| Time to maturity | $T$ | Expiry date - today (in years) |
| Risk-free rate | $r$ | Treasury bill yield (`^IRX`) |
| Volatility | $\sigma$ | Historical log returns of the stock |
| **Dividend schedule** | $D_i,\ t_i$ | KO's dividend history (projected forward) |

Plain Black-Scholes assumes the stock price is lognormal - smooth, no jumps. Fixed cash dividends break that assumption: on each ex-date the price drops by exactly $D_i$. So besides the usual inputs we need **two extra ones**: how much each dividend pays ($D_i$) and on what date ($t_i$).

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime
import scipy.stats as stats
from scipy.optimize import brentq

# Fetch 1 year of historical daily data for KO (a dividend-paying stock)
ticker_symbol = "KO"
df = yf.Ticker(ticker_symbol).history(period="1y")

# Extract the most recent closing price as the current stock price (S0)
S0 = df['Close'].iloc[-1]
print(f"Current {ticker_symbol} Stock Price (S0): ${S0:.2f}")

Current KO Stock Price (S0): $93.90


## Volatility from historical data

The model needs $\sigma$ = **annualized volatility** of the stock's log returns.

Daily log return:

$$r_t = \ln\!\left(\frac{P_t}{P_{t-1}}\right)$$

Log returns add up over time, so their standard deviation grows with the square root of time. With 252 trading days in a year:

$$\boxed{\sigma_{\text{annual}} = \operatorname{std}(r_t)\times\sqrt{252}}$$

**Assumption:** log returns are (roughly) normally distributed with constant volatility - then the stock price is lognormal, which is exactly what Black-Scholes assumes.

In [2]:
# Calculate daily log returns: ln(P_t / P_t-1)
df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))

# Drop missing values if any
log_returns = df['Log_Return'].dropna()

# standard deviation and annualization
daily_volatility = log_returns.std()
historical_volatility = daily_volatility * np.sqrt(252)

print(f"Daily Volatility: {daily_volatility:.6f}")
print(f"Annualized Historical Volatility (sigma): {historical_volatility:.4f} ({historical_volatility * 100:.2f}%)")

Daily Volatility: 0.011934
Annualized Historical Volatility (sigma): 0.1894 (18.94%)


## Option parameters

- **Strike** $K$ - we pick the nearest-dollar strike, so the option starts at-the-money
- **Time to maturity** $T$ - years until expiry: $T = \dfrac{\text{days to expiry}}{365.25}$
- **Risk-free rate** $r$ - continuously compounded (here from the 13-week T-bill index `^IRX`)
- **Volatility** $\sigma$ - from KO's historical log returns

The dividend schedule is estimated in the **next** section (it needs the expiry date, which is why it comes after this cell).

In [3]:
# Define option parameters

K = round(S0)                # nearest-dollar strike = at the money
expiry_date = datetime(2027, 6, 18)
today = datetime.now()

# ----------------------------------------------------------------------- #

# time to maturity (T) in years
T = (expiry_date - today).days / 365.25
print(f"Strike Price (K)          : ${K}")
print(f"Time to Maturity (T)      : {T:.8f} years ({(expiry_date - today).days} days)")

# Risk-Free Rate from ^IRX and historical volatility:
irx = yf.Ticker("^IRX")
# recent close price
r = irx.history(period="5d")['Close'].iloc[-1] / 100
sigma = historical_volatility
print(f"Risk-Free Rate (r)        : {r:.4f} ({r * 100:.2f}%)")
print(f"Volatility Used (sigma)   : {sigma:.4f} ({sigma * 100:.2f}%)")

Strike Price (K)          : $94
Time to Maturity (T)      : 0.65982204 years (241 days)
Risk-Free Rate (r)        : 0.0408 (4.08%)
Volatility Used (sigma)   : 0.1894 (18.94%)


## How to calculate the discrete dividends

**What is a discrete dividend?** A fixed amount of cash paid to shareholders on one specific ex-date - for example KO pays $\$0.46$ per share. On the morning the stock goes ex-dividend, the price drops **down by that exact amount**. Companies usually pay **4 times a year** (quarterly).

### Where the numbers come from

1. **Past payments** - `yf.Ticker("KO").dividends` returns every past dividend (ex-date + amount).
2. **Future payments** - not available for free, so we **project**: take the latest quarterly amount and repeat it every ~3 months until the option expires. *(In practice a trader would use the company's declared schedule - same idea, just official numbers.)*
3. **Only dividends before expiry matter**: keep payments with $0 < t_i < T$, where

$$t_i = \frac{\text{dividend date} - \text{today}}{365.25} \quad \text{(in years)}$$

### The two formulas we need for both models

**Present value today** (discount every payment back at the risk-free rate):

$$PV(D) = \sum_i D_i e^{-r t_i}$$

**Future value at expiry** (grow every payment forward to the expiry date):

$$FV(D) = \sum_i D_i e^{r (T - t_i)} = PV(D)\cdot e^{rT}$$

### example

$S_0 = 70$, one dividend $D = 0.50$ in 3 months ($t = 0.25$), $T = 1$ year, $r = 5\%$:

$$PV(D) = 0.50\,e^{-0.05 \times 0.25} \approx 0.4938, \qquad FV(D) = 0.50\,e^{0.05 \times 0.75} \approx 0.5127$$

The cell below prints the full schedule of our projected dividends with each one's present value, plus the totals $PV(D)$ and $FV(D)$.

In [7]:
# -----------------------------------------------------------------
# Build the dividend schedule:
#   1) take past payments from Yahoo Finance
#   2) project the latest quarterly amount forward until expiry
#   3) keep only the payments that fall before the option expires
# -----------------------------------------------------------------
tk = yf.Ticker(ticker_symbol)
div_series = tk.dividends                       # past dividends: (ex-date, amount)

div_schedule = []                               # (date, cash amount) expected before expiry

if len(div_series) > 0:
    if div_series.index.tz is not None:
        div_series.index = div_series.index.tz_localize(None)

    # average number of days between payments (KO pays quarterly ~ 91 days)
    gaps = div_series.index.to_series().diff().dt.days.dropna()
    period_days = float(gaps.median()) if len(gaps) > 0 else 91.25
    last_amount = float(div_series.iloc[-1])
    last_date = div_series.index[-1]

    # project the same cash amount forward until expiry
    d = last_date + pd.Timedelta(days=period_days)
    while d < expiry_date:
        if d > today:                           # ignore payments already in the past
            div_schedule.append((d, last_amount))
        d = d + pd.Timedelta(days=period_days)

    print(f"Last actual dividend : {last_date.date()}  ${last_amount:.4f}")
    print(f"Payment period used  : {period_days:.1f} days\n")
else:
    print(f"WARNING: no dividend history found for {ticker_symbol} - using an empty schedule.")

# schedule as (time in years, cash amount)
t_years = [(dt - today).days / 365.25 for dt, _ in div_schedule]
dividends = list(zip(t_years, [amt for _, amt in div_schedule]))

# present and future value of the dividends paid before expiry
PV_D = sum(a * np.exp(-r * t) for t, (dt, a) in zip(t_years, div_schedule))
FV_D = sum(a * np.exp(r * (T - t)) for t, (dt, a) in zip(t_years, div_schedule))

print("Dividend schedule before option expiry:")
print(f"{'#':>2}  {'Ex-date':<12} {'Amount':>8} {'t (years)':>10} {'PV':>8}")
for i, ((dt, a), t) in enumerate(zip(div_schedule, t_years), start=1):
    print(f"{i:>2}  {str(dt.date()):<12} {a:>8.4f} {t:>10.4f} {a * np.exp(-r * t):>8.4f}")
print("-" * 46)
print(f"Number of dividends : {len(dividends)}")
print(f"PV(D) = sum of D * e^(-r*t)  = ${PV_D:.4f}")
print(f"FV(D) = PV(D) * e^(r*T)      = ${FV_D:.4f}")

Last actual dividend : 2026-09-15  $0.5300
Payment period used  : 92.0 days

Dividend schedule before option expiry:
 #  Ex-date        Amount  t (years)       PV
 1  2026-12-16     0.5300     0.1561   0.5266
 2  2027-03-18     0.5300     0.4079   0.5213
----------------------------------------------
Number of dividends : 2
PV(D) = sum of D * e^(-r*t)  = $1.0479
FV(D) = PV(D) * e^(r*T)      = $1.0765


## The two dividend models: `'escrowed'` vs `'forward'`

When dividends are paid in **fixed cash amounts on known dates**, plain Black-Scholes does not apply exactly. Two classic approximations (overview in Frishling, 2002) simply adjust one input of the Black-Scholes formula:

### 1. Escrowed model (Black, 1975) - `dividend_model = 'escrowed'`

*Assumption: the stock price **minus the present value** of the dividends still to be paid follows a Geometric Brownian Motion.*

$$S^* = S_0 - \underbrace{\sum_i D_i e^{-r t_i}}_{PV(D)} \qquad \Longrightarrow \qquad C = BS(S^*,\ K,\ \dots)$$

**In simple words:** part of today's price is really "cash waiting to be paid out to shareholders". Take that cash out first - what remains (the risky part) behaves like a normal Black-Scholes stock. You replace $S_0$ with $S^*$ and keep the strike $K$ unchanged.

### 2. Forward model - `dividend_model = 'forward'`

*Assumption: the stock price **plus the forward value** of the dividends follows a Geometric Brownian Motion.*

$$K^* = K + \underbrace{\sum_i D_i e^{r (T - t_i)}}_{FV(D)} \qquad \Longrightarrow \qquad C = BS(S_0,\ K^*,\ \dots)$$

**In simple words:** keep today's price as it is, but whoever holds the stock until expiry also collects those dividends (money that could have earned interest). So the *effective strike* the buyer faces is **higher** by the future value of the dividends.

### Useful facts

- Only dividends with $0 < t_i < T$ (paid before expiry) matter.
- $FV(D) = PV(D)\cdot e^{rT}$ - the same money, just moved to the expiry date.
- Both models satisfy the **same European put-call parity** (see the next section) - but they give **different prices**, because each model splits the dividend effect differently (spot vs strike). Escrowed is the more common choice among practitioners.

## Put-call parity with discrete dividends (a free correctness check)

**In simple words:** a call and a put with the *same strike* and the *same expiry* are linked - one can be built from the other plus cash, adjusted for the cash dividends the stock pays before expiry.

**Why is it true?** Two portfolios that pay the **same amount at expiry** must cost the same today:

- **Portfolio A:** buy 1 call + keep $Ke^{-rT}$ cash in the bank. At expiry you end up with $\max(S_T, K)$.
- **Portfolio B:** buy 1 put + buy the stock, but set aside the present value of the dividends it will pay ($PV(D)$) - that cash is yours instead of the dividends. So today's cost of this side is $S_0 - PV(D)$, and at expiry you again end up with $\max(S_T, K)$.

Same payoff -> same price today:

$$C + Ke^{-rT} = P + S_0 - PV(D) \qquad\Longleftrightarrow\qquad \boxed{C - P = S_0 - PV(D) - Ke^{-rT}}$$

**Good to know:**

- Parity needs **no volatility and no model** - it follows from no-arbitrage alone, so it must hold *exactly*.
- The only difference vs the no-dividend version: the stock side shrinks from $S_0$ to $S_0 - PV(D)$, because the stock pays out that cash before expiry.
- **Both models (`'escrowed'` and `'forward'`) satisfy this identical parity**, even though their call and put prices differ - the printout below shows both `C - P` values next to the *should be* value. Residual ~0 = our formulas are coded correctly.

In [8]:
# Choose which dividend model to use: 'escrowed' or 'forward'
dividend_model = 'escrowed'

def bs_discrete(S0, K, T, r, sigma, dividends, model='escrowed'):
    """
    European call/put price with discrete cash dividends.
    dividends: list of (time_in_years, cash_amount) with 0 < t < T
    model = 'escrowed' : S* = S0 - PV(D), strike K unchanged
    model = 'forward'  : K* = K + FV(D), spot S0 unchanged
    """
    PV_D = sum(D * np.exp(-r * t) for t, D in dividends)
    FV_D = sum(D * np.exp(r * (T - t)) for t, D in dividends)

    if model == 'escrowed':
        S_adj, K_adj = S0 - PV_D, K
    elif model == 'forward':
        S_adj, K_adj = S0, K + FV_D
    else:
        raise ValueError("dividend_model must be 'escrowed' or 'forward'")

    d1 = (np.log(S_adj / K_adj) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    call = S_adj * stats.norm.cdf(d1) - K_adj * np.exp(-r * T) * stats.norm.cdf(d2)
    put = K_adj * np.exp(-r * T) * stats.norm.cdf(-d2) - S_adj * stats.norm.cdf(-d1)
    return call, put

# price with the selected model
call_price, put_price = bs_discrete(S0, K, T, r, sigma, dividends, dividend_model)

# price with BOTH models (and with no dividends = plain Black-Scholes) for comparison
call_esc, put_esc = bs_discrete(S0, K, T, r, sigma, dividends, 'escrowed')
call_fwd, put_fwd = bs_discrete(S0, K, T, r, sigma, dividends, 'forward')
call_none, put_none = bs_discrete(S0, K, T, r, sigma, [], 'escrowed')

# put-call parity (both models must give the same value)
parity = S0 - PV_D - K * np.exp(-r * T)

print("=" * 55)
print(f"Underlying Asset     : {ticker_symbol}")
print(f"Current Stock Price  : ${S0:.2f}")
print(f"Strike Price         : ${K}")
print(f"Expiration Date      : {expiry_date}    T = {T:.4f}")
print(f"Volatility Used      : {sigma * 100:.2f}%")
print(f"Dividends before exp.: {len(dividends)}   PV(D) = ${PV_D:.4f}   FV(D) = ${FV_D:.4f}")
print("-" * 55)
print(f"Selected model       : dividend_model = '{dividend_model}'")
print(f"  Call Option Price  : ${call_price:.2f}")
print(f"  Put Option Price   : ${put_price:.2f}")
print("-" * 55)
print("Comparison of both models:")
print(f"  escrowed (S* = S0 - PV(D) = {S0 - PV_D:.2f}) : Call ${call_esc:.2f}   Put ${put_esc:.2f}")
print(f"  forward  (K* = K + FV(D) = {K + FV_D:.2f})  : Call ${call_fwd:.2f}   Put ${put_fwd:.2f}")
print(f"  no dividends (plain Black-Scholes)      : Call ${call_none:.2f}   Put ${put_none:.2f}")
print("-" * 55)
print(f"Put-call parity escrowed : C - P = {call_esc - put_esc:.4f}")
print(f"Put-call parity forward  : C - P = {call_fwd - put_fwd:.4f}")
print(f"should be                :        = {parity:.4f}")
print("=" * 55)

Underlying Asset     : KO
Current Stock Price  : $93.90
Strike Price         : $94
Expiration Date      : 2027-06-18 00:00:00    T = 0.6598
Volatility Used      : 18.94%
Dividends before exp.: 2   PV(D) = $1.0479   FV(D) = $1.0765
-------------------------------------------------------
Selected model       : dividend_model = 'escrowed'
  Call Option Price  : $6.35
  Put Option Price   : $5.01
-------------------------------------------------------
Comparison of both models:
  escrowed (S* = S0 - PV(D) = 92.85) : Call $6.35   Put $5.01
  forward  (K* = K + FV(D) = 95.08)  : Call $6.42   Put $5.07
  no dividends (plain Black-Scholes)      : Call $6.96   Put $4.57
-------------------------------------------------------
Put-call parity escrowed : C - P = 1.3471
Put-call parity forward  : C - P = 1.3471
should be                :        = 1.3471


## Implied volatility under both models

Same idea as always: we know a *market price* and search for the $\sigma$ that would produce it,

$$BS(\sigma) = \text{market price},$$

solved with **Brent's method** (`scipy.optimize.brentq`). The pricer used by the root-finder contains the same dividend adjustment as the selected model.

Because the two models price the same option slightly differently, they also produce **two different implied volatilities**. If a trader forgets to adjust for dividends, part of the "volatility" they see is really just the dividend effect.

In [10]:
def bs_price_discrete(CP, S0, K, sigma, T, r, dividends, model):
    """Black-Scholes price with discrete dividends, used by the root-finder."""
    if sigma <= 0:
        return 0.0
    PV_D = sum(D * np.exp(-r * t) for t, D in dividends)
    FV_D = sum(D * np.exp(r * (T - t)) for t, D in dividends)

    if model == 'escrowed':
        S_adj, K_adj = S0 - PV_D, K
    else:
        S_adj, K_adj = S0, K + FV_D

    d1 = (np.log(S_adj / K_adj) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if CP.upper() == 'CALL':
        return stats.norm.cdf(d1) * S_adj - stats.norm.cdf(d2) * K_adj * np.exp(-r * T)
    return stats.norm.cdf(-d2) * K_adj * np.exp(-r * T) - stats.norm.cdf(-d1) * S_adj

def implied_volatility(market_price, CP, S0, K, T, r, dividends, model):
    """
    Finds the implied volatility by reverse-engineering the model price
    using Brent's root-finding method.
    """
    # objective function: model price - market price
    def objective_function(sigma):
        return bs_price_discrete(CP, S0, K, sigma, T, r, dividends, model) - market_price

    try:
        # search for volatility between 0.001 (0.1%) and 5.0 (500%)
        iv = brentq(objective_function, 1e-3, 5.0)
        return iv
    except ValueError:
        # returns NaN if the market price is invalid
        return np.nan


market_price_call = 4.50    # observed market prices
market_price_put = 3.50

iv = {}
for model in ['escrowed', 'forward']:
    iv[(model, 'CALL')] = implied_volatility(market_price_call, 'CALL', S0, K, T, r, dividends, model)
    iv[(model, 'PUT')] = implied_volatility(market_price_put, 'PUT', S0, K, T, r, dividends, model)

print()
print(f"Market prices used : call ${market_price_call:.2f}, put ${market_price_put:.2f}")
print()
print(f"Implied Volatility_CALL (escrowed): {iv[('escrowed', 'CALL')]:.4f} ({iv[('escrowed', 'CALL')] * 100:.2f}%)")
print(f"Implied Volatility_PUT  (escrowed): {iv[('escrowed', 'PUT')]:.4f} ({iv[('escrowed', 'PUT')] * 100:.2f}%)")
print()
print(f"Implied Volatility_CALL (forward) : {iv[('forward', 'CALL')]:.4f} ({iv[('forward', 'CALL')] * 100:.2f}%)")
print(f"Implied Volatility_PUT  (forward) : {iv[('forward', 'PUT')]:.4f} ({iv[('forward', 'PUT')] * 100:.2f}%)")


Market prices used : call $4.50, put $3.50

Implied Volatility_CALL (escrowed): 0.1269 (12.69%)
Implied Volatility_PUT  (escrowed): 0.1386 (13.86%)

Implied Volatility_CALL (forward) : 0.1255 (12.55%)
Implied Volatility_PUT  (forward) : 0.1371 (13.71%)
